In [13]:
pip install gensim

  Using cached smart_open-7.5.0-py3-none-any.whl.metadata (24 kB)
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 659.8 kB/s eta 0:00:37
   - -------------------------------------- 0.8/24.4 MB 826.1 kB/s eta 0:00:29
   - -------------------------------------- 0.8/24.4 MB 826.1 kB/s eta 0:00:29
   - -------------------------------------- 0.8/24.4 MB 826.1 kB/s eta 0:00:29
   - -------------------------------------- 1.0/24.4 MB 688.2 kB/s eta 0:00:34
   - -------------------------------------- 1.0/24.4 MB 688.2 kB/s eta 0:00:34
   - -------------------------------------- 1.0/24.4 MB 688.2 kB/s eta 0:00:34
   - -------------------------------------- 1.0/24.4 MB 688.2 kB/s eta 0:00:34
   -- ----------------

In [18]:
# !pip install scikit-learn pandas numpy underthesea

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from sklearn.feature_extraction.text import TfidfVectorizer

In [19]:
df = pd.read_csv("ML/data/vn_fake_news_combined.csv")

df.head()

,text,label
0,Tiền vệ Bruno Fernandes đeo băng đội trưởng th...,0
1,SingaporeMelissa Lim 36 ung_thư vú giai_đoạn 2...,0
2,"Bác sĩ Phạm Thái Ngọc Long, Khoa Cấp cứu, Bệnh...",0
3,khí_hậu chủ_yếu hoa kỳ bất_ổn xã_hội bất_công ...,0
4,TP HCMÔng Bá 60 trò_chuyện đột_ngột im_lặng li...,0


In [21]:
df['style_label'] = df['label']  # giả định hợp lý

df[['text', 'style_label']].head()

,text,style_label
0,Tiền vệ Bruno Fernandes đeo băng đội trưởng th...,0
1,SingaporeMelissa Lim 36 ung_thư vú giai_đoạn 2...,0
2,"Bác sĩ Phạm Thái Ngọc Long, Khoa Cấp cứu, Bệnh...",0
3,khí_hậu chủ_yếu hoa kỳ bất_ổn xã_hội bất_công ...,0
4,TP HCMÔng Bá 60 trò_chuyện đột_ngột im_lặng li...,0


In [22]:
SHOCK_WORDS = [
    "tin chấn động", "chấn động", "sốc", "sốc nặng",
    "không thể tin", "không tin nổi",
    "kinh hoàng", "bàng hoàng",
    "nhân loại được cứu"
]

ABSOLUTE_WORDS = [
    "100%", "chắc chắn", "hoàn toàn",
    "ai cũng biết", "tất cả đều",  "duy nhất", "cam kết", "đảm bảo",
    "sự thật là", "không thể sai"
]

AGGRESSIVE_WORDS = [
    "tẩy chay", "vạch mặt", "lật mặt",
    "phải lên án", "đáng lên án", "không thể chấp nhận", "tẩy chay"
]

FAKE_SOURCE_PHRASES = [
    "bạn tôi là bác sĩ",
    "nguồn tin nội bộ",
    "kênh truyền hình lớn",
    "báo nước ngoài đưa tin",
    "báo mỹ đưa tin"
]

FEAR_WORDS = [
    "chết người",
    "bùng phát", "dịch bệnh", "không thể cứu"
]

ANTI_STATE_WORDS = [
    "chế độ cộng sản", "nhà nước độc tài",
    "đảng độc quyền", "chính quyền thối nát","đảng cầm quyền thao túng"
]

SOCIAL_UNREST_WORDS = [
    "dân khổ", "quan tham",
    "bóc lột nhân dân","không còn tương lai"
]

CALL_TO_ACTION_WORDS = [
    "phải đứng lên", "toàn dân vùng lên",
    "lật đổ", "không thể im lặng"
]

DEMOCRACY_MISUSE_WORDS = [
    "vi phạm nhân quyền",
    "bịt miệng dân",
    "đàn áp người dân",
    "không có tự do ngôn luận"
]

In [23]:
def extract_style_features(text):
    text = str(text)
    text_lower = text.lower()

    return {
        # Cấu trúc
        "length": len(text),
        "caps_ratio": sum(1 for c in text if c.isupper()) / max(len(text), 1),
        "exclamation_count": text.count("!"),
        "question_count": text.count("?"),

        # Thao túng cảm xúc
        "shock_words": sum(w in text_lower for w in SHOCK_WORDS),
        "absolute_words": sum(w in text_lower for w in ABSOLUTE_WORDS),
        "aggressive_words": sum(w in text_lower for w in AGGRESSIVE_WORDS),
        "fear_words": sum(w in text_lower for w in FEAR_WORDS),

        # Mạo danh nguồn
        "fake_source_claims": sum(w in text_lower for w in FAKE_SOURCE_PHRASES),

        # Chính trị – kích động
        "anti_state": sum(w in text_lower for w in ANTI_STATE_WORDS),
        "social_unrest": sum(w in text_lower for w in SOCIAL_UNREST_WORDS),
        "call_to_action": sum(w in text_lower for w in CALL_TO_ACTION_WORDS),
        "democracy_misuse": sum(w in text_lower for w in DEMOCRACY_MISUSE_WORDS),
    }

In [24]:
print("⏳ Đang trích xuất style features...")

style_features = df['text'].apply(extract_style_features)
style_df = pd.DataFrame(list(style_features))

style_df.head()

⏳ Đang trích xuất style features...


,length,caps_ratio,exclamation_count,question_count,shock_words,absolute_words,aggressive_words,fear_words,fake_source_claims,anti_state,social_unrest,call_to_action,democracy_misuse
0,2136,0.044476,0,0,0,1,0,0,0,0,0,0,0
1,3610,0.030471,0,0,0,0,0,1,0,0,0,0,0
2,2654,0.048983,0,0,0,1,0,0,0,0,0,0,0
3,287,0.000000,0,0,0,0,0,0,0,0,0,0,0
4,2085,0.041247,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Lấy top fake news keywords
top_fake_keywords = df_fake_keywords[['keyword', 'fake_count', 'real_count', 'log_odds']].head(50)

print("📌 Top 50 từ khóa đặc trưng cho tin giả (dựa trên log odds):\n")
print(top_fake_keywords.to_string(index=False))

# Xuất ra danh sách từ khóa
fake_key_phrases = top_fake_keywords['keyword'].tolist()

print("\n" + "="*60)
print(f"✅ Đã trích xuất {len(fake_key_phrases)} từ/cụm từ đặc trưng cho tin giả")
print("="*60)

📌 Top 50 từ khóa đặc trưng cho tin giả (dựa trên log odds):

            keyword  fake_count  real_count  log_odds
      đại kỷ_nguyên         831           0  6.723832
   sức_khỏe làm_đẹp         749           0  6.620073
            báo đại         747           0  6.617403
chuyên_mục sức_khỏe         742           0  6.610696
    y_học nhận_thức         742           0  6.610696
  khoa_học thân_thể         741           0  6.609349
  thân_thể rộng_lớn         740           0  6.608001
     rộng_lớn y_học         740           0  6.608001
   chính_xác bổ_ích         740           0  6.608001
   bổ_ích tuy_nhiên         740           0  6.608001
      tham_khảo mọi         739           0  6.606650
     viết tính_chất         739           0  6.606650
      mọi chẩn_đoán         739           0  6.606650
     làm_đẹp ý_kiến         739           0  6.606650
     nỗ_lực bạn_đọc         738           0  6.605298
   kỷ_nguyên nỗ_lực         736           0  6.602588
        phép có_thể  